# Sentiment Analysis of Customer Feedback using BERT

## Objective
In this project, we aim to classify customer feedback into positive or negative sentiment using a BERT-based deep learning model. This can help automate customer sentiment tracking and improve support quality monitoring.

We’ll use the **Customer Feedback Dataset** from Kaggle, which contains short feedback messages and their associated sentiment labels.

---

## 1. Environment Setup

Before continuing, we verify the Python environment and install any missing libraries. We'll be using:
- Python 3.8+
- PyTorch
- Hugging Face Transformers
- pandas
- scikit-learn (for evaluation)

> Note: This notebook is intended to run inside VSCode using the Jupyter extension, with a virtual environment activated.


In [75]:
import sys
import torch
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

# For Colab users: install required packages
# Uncomment the line below if running in Google Colab
# !pip install transformers pandas scikit-learn torch


/Users/luis/Documents/tech-support-data-science-projects/project_04_sentiment_analysis_bert/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python version: 3.13.2 (v3.13.2:4f8bb3947cf, Feb  4 2025, 11:51:10) [Clang 15.0.0 (clang-1500.3.9.4)]
PyTorch version: 2.8.0
GPU available: False


## 2. Load and Explore the Dataset

We will use the [Customer Feedback Dataset](https://www.kaggle.com/datasets/vishweshsalodkar/customer-feedback-dataset) from Kaggle, which contains short customer feedback texts labeled by sentiment (positive or negative).

Before training the model, we will:
- Load the dataset using `pandas`
- Check for missing or duplicated values
- Preview class distribution
- Look at a few example feedback entries


### Preprocessing Steps Applied to the CSV Data

**Raw Data Format Example:**

```csv
"Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score"
"""I love this product!"", Positive, Twitter, 2023-06-15 09:23:14, @user123, New York, 0.85"
"""The service was terrible."", Negative, Yelp Reviews, 2023-06-15 11:45:32, user456, Los Angeles, 0.65"
```

As shown above, each row is enclosed in double quotes, and the text field itself uses double double-quotes (`""`) for embedded quotes. This non-standard format causes issues with standard CSV parsers, so the following preprocessing steps are required:

- **Manual Parsing:** Each line is read and stripped of leading/trailing whitespace.
- **Quote Handling:** Outer double quotes are removed from lines, and the `csv.reader` is used to correctly parse fields containing commas or embedded quotes.
- **Header Extraction:** The first row is used as the column header, and subsequent rows are treated as data.
- **DataFrame Construction:** The parsed rows are assembled into a pandas DataFrame with cleaned column names.
- **Text Cleanup:** Leading and trailing double quotes in the "Text" column are removed using regular expressions.
- **Missing Data Handling:** Empty lines and rows with missing values are skipped or cleaned to ensure consistency.

These steps ensure the data is structured and ready for analysis and modeling.

In [76]:
import pandas as pd
import csv

csv_data = "../data/raw/customer_feedback.csv"

rows = []
with open(csv_data, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # skip empty lines
            # Remove outer quotes if present
            if line.startswith('"') and line.endswith('"'):
                line = line[1:-1]
            # Use csv.reader to parse the inner CSV
            reader = csv.reader([line], skipinitialspace=True)
            rows.append(next(reader))

# Create DataFrame
header = rows[0]
data = rows[1:]
customer_feedback = pd.DataFrame(data, columns=[h.strip() for h in header])

# Clean up leading/trailing double quotes in the Text column
customer_feedback["Text"] = customer_feedback["Text"].str.replace('^""', '', regex=True)
customer_feedback["Text"] = customer_feedback["Text"].str.replace('""$', '', regex=True)

customer_feedback = customer_feedback.dropna(how="all")

print(f"Dataset shape: {customer_feedback.shape}")
customer_feedback.head()

Dataset shape: (96, 7)


,Text,Sentiment,Source,Date/Time,User ID,Location,Confidence Score
0,I love this product!,Positive,Twitter,2023-06-15 09:23:14,@user123,New York,0.85
1,The service was terrible.,Negative,Yelp Reviews,2023-06-15 11:45:32,user456,Los Angeles,0.65
2,This movie is amazing!,Positive,IMDb,2023-06-15 14:10:22,moviefan789,London,0.92
3,I'm so disappointed with their customer support.,Negative,Online Forum,2023-06-15 17:35:11,forumuser1,Toronto,0.78
4,Just had the best meal of my life!,Positive,TripAdvisor,2023-06-16 08:50:59,foodie22,Paris,0.88


Above we can see the cleaned data separated in columns, ready for processing. Let's see if have any missing values.

In [77]:
# Check for missing values
customer_feedback.isnull().sum()


Text                0
Sentiment           0
Source              0
Date/Time           0
User ID             0
Location            0
Confidence Score    0
dtype: int64

## 3. Preprocessing and Label Encoding

Before we feed the data into a BERT model, we need to:
- Clean the text minimally (optional — BERT can handle raw text fairly well)
- Encode the sentiment labels into numeric form:
  - `positive` → 1
  - `negative` → 0

Since BERT uses its own tokenizer, we **do not need to lowercase, remove punctuation**, or apply stemming/lemmatization — the tokenizer handles it internally.


In [78]:
# Encode sentiment labels: Positive -> 1, Negative -> 0
customer_feedback['label'] = customer_feedback['Sentiment'].map({'Positive': 1, 'Negative': 0})

# Minimal text cleanup (optional for BERT, but let's strip whitespace)
customer_feedback['Text'] = customer_feedback['Text'].str.strip()

# Show label distribution
print(customer_feedback['label'].value_counts())
customer_feedback[['Text', 'Sentiment', 'label']].head(20)

label
1    53
0    43
Name: count, dtype: int64


,Text,Sentiment,label
0,I love this product!,Positive,1
1,The service was terrible.,Negative,0
2,This movie is amazing!,Positive,1
3,I'm so disappointed with their customer support.,Negative,0
4,Just had the best meal of my life!,Positive,1
5,The quality of this product is subpar.,Negative,0
6,I can't stop listening to this song. It's incr...,Positive,1
7,Their website is so user-friendly. Love it!,Positive,1
8,I loved the movie! It was fantastic!,Positive,1
9,The customer service was terrible.,Negative,0


In [79]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    list(customer_feedback['Text']),
    list(customer_feedback['label']),
    test_size=0.2,
    random_state=42
)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

# convert to torch dataset
class CustomerFeedbackDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CustomerFeedbackDataset(train_encodings, train_labels)
val_dataset = CustomerFeedbackDataset(val_encodings, val_labels)


In [80]:
# Load BERT model for sequence classification (binary: 2 labels)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Fine-tune BERT
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# Evaluate the fine-tuned BERT model on the validation set
eval_results = trainer.evaluate(eval_dataset=val_dataset)
print(eval_results)

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()
    sentiment = "Positive" if predicted_class == 1 else "Negative"
    return sentiment

# Example usage:
custom_text = "The support team was very helpful and resolved my issue quickly."
print(f"Sentiment: {predict_sentiment(custom_text)}")